# Worldle Final Project

This completed reference notebook builds a small Worldle-style country guessing game using the repo data, `ipyleaflet`, `ipywidgets`, and the updated `wdo` package.

## What was implemented

- `wdo.geometry.bbox.bbox_from_feature` and `bbox_from_features`
- `wdo.maps.leaflet_helpers.make_map`, `add_geojson`, `fit_map_to_geojson`, controls, paths, and bboxes
- `wdo.games.worldle.choose_target`, `feature_center`, `guess_feedback`, and `format_feedback`
- `wdo.io.country_lookup.build_country_lookup` for the ISO-3 polygon data to ISO-2 flag data bridge

## Known notes

The center calculation defaults to a bounding-box center. It is fast and good enough for a classroom game, but countries with far-flung territories or antimeridian geometry can produce imperfect direction clues.

## Completed round screenshot

![Completed Worldle round](worldle_completed_round.png)



In [1]:
from pathlib import Path
import json
import base64

from IPython.display import display
from ipywidgets import Button, Combobox, HBox, HTML, Output, VBox

from wdo.games.worldle import choose_target, guess_feedback, format_feedback
from wdo.io.country_lookup import build_country_lookup
from wdo.maps.leaflet_helpers import add_geojson, fit_map_to_geojson, make_map


In [2]:
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "Resources" / "Data" / "countries.geojson").exists():
    ROOT = ROOT.parent

COUNTRIES_PATH = ROOT / "Resources" / "Data" / "countries.geojson"
FLAGS_DIR = ROOT / "Resources" / "Data" / "flag-icons"
FLAGS_INDEX = FLAGS_DIR / "country.json"

countries = json.loads(COUNTRIES_PATH.read_text(encoding="utf-8"))
flag_index = json.loads(FLAGS_INDEX.read_text(encoding="utf-8"))
lookup = build_country_lookup(countries, flag_index)

features = [item["feature"] for item in lookup.values()]
name_to_iso3 = {item["name"]: iso3 for iso3, item in lookup.items()}
country_names = sorted(name_to_iso3)

print(f"Loaded {len(features)} countries")


Loaded 237 countries


In [3]:
def flag_img(flag_path, width=34):
    if not flag_path:
        return ""
    path = FLAGS_DIR / flag_path
    if not path.exists():
        return ""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f'<img src="data:image/svg+xml;base64,{encoded}" width="{width}" style="vertical-align:middle;border:1px solid #ddd">'


def render_guess_row(country_name, flag_path, feedback):
    distance = feedback["distance_km"]
    color = "#16a34a" if distance < 500 else "#ca8a04" if distance < 2500 else "#dc2626"
    return f"""
    <div style="display:flex;align-items:center;gap:10px;font-family:system-ui,sans-serif;padding:6px 0;border-bottom:1px solid #eee">
      <span style="width:40px">{flag_img(flag_path)}</span>
      <strong style="min-width:180px">{country_name}</strong>
      <span style="font-size:24px;width:32px;text-align:center">{feedback['arrow']}</span>
      <span style="color:{color};font-weight:700">{distance:,.0f} km</span>
      <span style="color:#666">toward {feedback['compass']}</span>
    </div>
    """


In [4]:
class WorldleGame:
    def __init__(self, features, lookup, seed=4543, max_guesses=6):
        self.features = list(features)
        self.lookup = lookup
        self.by_iso3 = {iso3: item["feature"] for iso3, item in lookup.items()}
        self.target = choose_target(self.features, seed=seed)
        self.target_iso3 = self.target["properties"].get("ISO3166-1-Alpha-3") or self.target["properties"].get("ISO_A3")
        self.target_name = self.target["properties"].get("name") or self.target["properties"].get("ADMIN")
        self.guesses = []
        self.max_guesses = max_guesses
        self.finished = False

    def submit_guess(self, iso3):
        if self.finished:
            return {"message": "Game is already finished."}
        guess = self.by_iso3[iso3]
        feedback = guess_feedback(guess, self.target)
        feedback["iso3"] = iso3
        self.guesses.append(feedback)
        if feedback["correct"] or len(self.guesses) >= self.max_guesses:
            self.finished = True
        return feedback

    @property
    def guesses_left(self):
        return max(self.max_guesses - len(self.guesses), 0)


In [5]:
game = WorldleGame(features, lookup, seed=4543, max_guesses=6)

world_map = make_map(center=(20, 0), zoom=2)
add_geojson(
    world_map,
    {"type": "FeatureCollection", "features": [game.target]},
    name="Mystery country",
    style={"color": "#111827", "fillColor": "#f59e0b", "weight": 2, "fillOpacity": 0.65},
)
fit_map_to_geojson(world_map, {"type": "FeatureCollection", "features": [game.target]})

selector = Combobox(
    placeholder="Type a country name",
    options=country_names,
    description="Guess:",
    ensure_option=True,
    layout={"width": "420px"},
)
guess_button = Button(description="Guess", button_style="primary")
give_up_button = Button(description="Give up", button_style="warning")
new_game_button = Button(description="New seed", button_style="")
banner = HTML(f"<strong>Guesses left:</strong> {game.guesses_left}")
history = Output()


def reveal_target(text=None):
    banner.value = text or f"<strong>Target:</strong> {game.target_name}"


def on_guess(_=None):
    if game.finished or not selector.value:
        return
    iso3 = name_to_iso3[selector.value]
    feedback = game.submit_guess(iso3)
    item = lookup[iso3]
    with history:
        display(HTML(render_guess_row(item["name"], item.get("flag_path"), feedback)))
    if feedback["correct"]:
        reveal_target(f"<strong style='color:#16a34a'>Correct:</strong> {game.target_name}")
    elif game.finished:
        reveal_target(f"<strong style='color:#dc2626'>Out of guesses.</strong> Target: {game.target_name}")
    else:
        banner.value = f"<strong>Guesses left:</strong> {game.guesses_left}"


def on_give_up(_=None):
    game.finished = True
    reveal_target()


guess_button.on_click(on_guess)
give_up_button.on_click(on_give_up)

ui = VBox([
    world_map,
    HBox([selector, guess_button, give_up_button]),
    banner,
    history,
])

ui


## Reflection

This notebook keeps the game logic in `wdo` and uses the notebook mostly as a UI layer. The main data mismatch was country polygons using ISO-3 codes while flag files use ISO-2 codes, so the lookup helper joins by country name and keeps countries usable even when a flag match is missing. The biggest known limitation is the center calculation: bounding-box centers are simple, but they are not guaranteed to fall inside odd-shaped countries. A stronger version would use a representative point algorithm for polygons and multipolygons.